## A1. Imports and settings

In [ ]:
import importlib
from pathlib import Path
import pandas as pd
import sys
import subprocess
from check_number_of_files import count_folders, check_num_dcm_report, patients_with_more_than_1_txt
from labelling.Build_disease_vocab import *
from labelling.scan_disease_v2 import main
import labelling.count_positive_cases as counter

importlib.reload(counter)
count_main = counter.main
from labelling.build_prognosis_labels import main as prognosis_main


### A1.1. Path declaration to all directories

In [68]:
path1 = Path(r"C:\Users\Zila\Downloads\BN5212\MIMIC-CXR\p10\p10")
project_dir = Path(r"C:\Users\Zila\Downloads\BN5212\midterm_prognosis")
labelling_dir = project_dir / "labelling"
vocab_path = labelling_dir / "disease_vocab" / "unique_diseases.csv"
patients_path = labelling_dir / "cxr_labels" / "patients_with_more_than_1_txt.csv"
output_path = labelling_dir / "cxr_labels_selected_everpositive"
prognosis_path = labelling_dir / "prognosis_labels"
hybrid_path = labelling_dir / "prognosis_labels_hybrid"

## A2. Checking the count of Patients, Dicom x-ray images, and Reports

In [4]:
print("Folders in MIMIC-cxr path:", count_folders(path1))
check_num_dcm_report(path1)

Folders in MIMIC-cxr path: 1000
p10000032: 7 DICOM images, 4 reports
p10000764: 3 DICOM images, 1 reports
p10000898: 5 DICOM images, 2 reports
p10000935: 10 DICOM images, 6 reports
p10000980: 16 DICOM images, 9 reports
p10001038: 3 DICOM images, 1 reports
p10001122: 5 DICOM images, 2 reports
p10001176: 5 DICOM images, 2 reports
p10001217: 3 DICOM images, 2 reports
p10001401: 10 DICOM images, 6 reports
p10001851: 2 DICOM images, 1 reports
p10001884: 55 DICOM images, 31 reports
p10002013: 15 DICOM images, 9 reports
p10002131: 2 DICOM images, 1 reports
p10002157: 2 DICOM images, 1 reports
p10002177: 2 DICOM images, 1 reports
p10002221: 2 DICOM images, 1 reports
p10002428: 34 DICOM images, 28 reports
p10002430: 9 DICOM images, 4 reports
p10002557: 2 DICOM images, 1 reports
p10002559: 5 DICOM images, 2 reports
p10002661: 2 DICOM images, 1 reports
p10002804: 2 DICOM images, 1 reports
p10002930: 2 DICOM images, 1 reports
p10003019: 30 DICOM images, 17 reports
p10003052: 3 DICOM images, 1 repo

### A2.1. Out of those patients, how many have more than 1 txt file? (i.e. more than 1 report)

In [13]:
count, patient_list = patients_with_more_than_1_txt(path1)
print(f"Patients with more than 1 x-ray visits: {count}")


Patients with more than 1 x-ray visits: 506


In [10]:
patient_list = pd.DataFrame(patient_list, columns=["Patient"])
patient_list.head()

,Patient
0,p10000032
1,p10000898
2,p10000935
3,p10000980
4,p10001122


In [11]:
patient_list.to_csv("patients_with_more_than_1_txt.csv", index=False)

#to do: add figure like the one generated by sahana to see the plot count

## A3. From those patients, we will label based on what disease they have

### A3.1. Firstly, we need to build the disease vocab. We obtained it from d_icd_diagnoses.csv in MIMIC-IV

In [14]:
diagnosis_path = Path("C:\\Users\\Zila\\Downloads\\BN5212\\MIMIC-IV\\3.1\\hosp\\d_icd_diagnoses.csv")

In [18]:
dia_df= load_icd_diagnoses(diagnosis_path)
mapping, labels = load_icd_to_phecode_mapping()
mapped = map_icd_to_phecode(dia_df, mapping, labels)
disease_list = drop_unmapped(mapped)

In [19]:
print(f"Unique mapped disease/phenotype names: {len(disease_list)}")
print(disease_list[:30])

Unique mapped disease/phenotype names: 3544
['(Idiopathic) normal pressure hydrocephalus', '3-methylglutaconic aciduria*', '46, XX/46, XY*', 'ABO isoimmunization of newborn', 'ANCA-associated vasculitides*', 'Abdominal aortic aneurysm', 'Abdominal migraine*', 'Abdominal or pelvic swelling, mass, or lump', 'Abdominal pain', 'Abdominal pregnancy', 'Abdominal rigidity', 'Abdominal tenderness', 'Abnormal Papanicolaou smear', 'Abnormal auditory function study', 'Abnormal blood-gas level', 'Abnormal bowel sounds and visible peristalsis', 'Abnormal cardiac sounds', 'Abnormal coagulation profile', 'Abnormal cytological findings in specimens from genital organs', 'Abnormal digit morphology', 'Abnormal electrocardiogram [ECG] [EKG]', 'Abnormal electroencephalogram [EEG]', 'Abnormal findings on antenatal screening of mother', 'Abnormal findings on diagnostic imaging of heart and coronary circulation', 'Abnormal findings on diagnostic imaging of lung', 'Abnormal findings on diagnostic imaging of s

### A3.2. Save the unique diseases vocab into a csv file

In [21]:
pd.DataFrame({"disease": disease_list}).to_csv("labelling/disease_vocab/unique_diseases.csv", index=False)

### A3.1. Now, we scan the reports in cxr that has any of the disease in the vocab we built

In [34]:
main([
    "--vocab", str(vocab_path),
    "--patients", str(patients_path),
    "--reports", str(path1),
    "--skip-uncertain-after-positive",
    "--out", str(output_path),
])

Patient filter: 506 requested, 506 with reports found.
Reading dates from DICOM headers (image pixels are not loaded).
Reports without dates: 0; these remain eligible for review.
Matcher: regex fallback (install flashtext for faster matching); vocabulary: 3544; reports: 2857
Scanned 500/2857
Scanned 1000/2857
Scanned 1500/2857
Scanned 2000/2857
Scanned 2500/2857
Scanned 2857/2857
Saved results to C:\Users\Zila\Downloads\BN5212\midterm_prognosis\labelling\cxr_labels_selected_everpositive
Uncertain reviews skipped after an earlier positive: 78
Labels are sparse: missing rows are NOT negative labels. Check report_summary.
Review samples of positives, negatives, and no_matches as well as review_queue.


### A3.2. Afterwards, we look at the positive diseases labels, and pick 5 of the most abundant

In [40]:
labels_path = output_path / "study_labels.csv"
reviews_path = output_path / "review_queue.csv"
counts_path = output_path / "disease_positive_counts.csv"

In [47]:
count_main([
    "--labels", str(labels_path),
    "--reviews", str(reviews_path),
    "--vocab", str(vocab_path),
    "--out", str(counts_path),
])

Disease                                              Patients    Reports
Pleural effusion                                          167        415
Cardiomegaly                                              155        364
Edema                                                     139        363
Pneumonia                                                 130        225
Pulmonary edema                                           119        277
Fluid overload                                             39         44
Fractures                                                  29         61
Emphysema                                                  28         54
Scoliosis                                                  18         33
Hernia                                                     14         28
Hypertension                                                9          9
Term                                                        8          8
Loss of height                                     

### A3.2. Out of those 5 diseases, we now label the prognosis into improving, stable, worsening, mixed, or no_comparison

In [ ]:
prognosis_main([
    "--reports", str(path1),
    "--patients", str(patients_path),
    "--out", str(prognosis_path),
])

Read 100/2857 reports
Read 200/2857 reports
Read 300/2857 reports
Read 400/2857 reports
Read 500/2857 reports
Read 600/2857 reports
Read 700/2857 reports
Read 800/2857 reports
Read 900/2857 reports
Read 1000/2857 reports
Read 1100/2857 reports
Read 1200/2857 reports
Read 1300/2857 reports
Read 1400/2857 reports
Read 1500/2857 reports
Read 1600/2857 reports
Read 1700/2857 reports
Read 1800/2857 reports
Read 1900/2857 reports
Read 2000/2857 reports
Read 2100/2857 reports
Read 2200/2857 reports
Read 2300/2857 reports
Read 2400/2857 reports
Read 2500/2857 reports
Read 2600/2857 reports
Read 2700/2857 reports
Read 2800/2857 reports
Saved 14285 disease-study rows and 11755 candidate targets to C:\Users\Zila\Downloads\BN5212\midterm_prognosis\labelling\prognosis_labels
Verify comparison references and fill reviewed_target before training. Split by subject_id.


In [57]:
diseases = [
    "Pleural effusion",
    "Cardiomegaly",
    "Edema",
    "Pneumonia",
    "Pulmonary edema",
]

In [ ]:
prognosis.main([
    "--reports", str(path1),
    "--patients", str(patients_path),
    "--labels", str(labels_path),
    "--reviews", str(reviews_path),
    "--mode", "lenient",
    "--out", str(hybrid_path),
])

Read 100/2857 reports
Read 200/2857 reports
Read 300/2857 reports
Read 400/2857 reports
Read 500/2857 reports
Read 600/2857 reports
Read 700/2857 reports
Read 800/2857 reports
Read 900/2857 reports
Read 1000/2857 reports
Read 1100/2857 reports
Read 1200/2857 reports
Read 1300/2857 reports
Read 1400/2857 reports
Read 1500/2857 reports
Read 1600/2857 reports
Read 1700/2857 reports
Read 1800/2857 reports
Read 1900/2857 reports
Read 2000/2857 reports
Read 2100/2857 reports
Read 2200/2857 reports
Read 2300/2857 reports
Read 2400/2857 reports
Read 2500/2857 reports
Read 2600/2857 reports
Read 2700/2857 reports
Read 2800/2857 reports
Saved 14285 disease-study rows and 11755 candidate targets to C:\Users\Zila\Downloads\BN5212\midterm_prognosis\labelling\prognosis_labels_hybrid
Verify comparison references and fill reviewed_target before training. Split by subject_id.


In [67]:
pairs = pd.read_csv(hybrid_path / "prognosis_pairs.csv")

existing_disease = pairs[
    pairs["input_disease_state"].eq("positive")
]

display(pd.crosstab(
    existing_disease["disease"],
    existing_disease["candidate_target"],
).reindex(
    columns=["improving", "stable", "worsening", "mixed", "no_comparison"],
    fill_value=0,
))

candidate_target,improving,stable,worsening,mixed,no_comparison
disease,,,,,
Cardiomegaly,4,82,14,1,208
Edema,64,20,25,2,209
Pleural effusion,58,54,47,12,190
Pneumonia,27,4,9,1,144
Pulmonary edema,53,16,20,1,156


## A4. Now that we have labeled, we will load training, validation, and test sets


In [71]:
from data_loader.cxr_prognosis_dataloader import make_dataloaders

loaders = make_dataloaders(
    pairs_csv=hybrid_path / "prognosis_pairs.csv",
    disease="Pleural effusion",
    reports_root=path1,
    batch_size=16,
    image_size=224,
    num_workers=0,  # Suitable for Windows notebooks
    seed=42,
    positive_input_only=True,
    use_candidates=True,  # Exploratory baseline using automatic labels
    split_csv=labelling_dir / "prognosis_patient_splits.csv",
    manifest_path=hybrid_path / "pleural_effusion_image_manifest.csv",
)

train_loader = loaders["train"]
val_loader = loaders["val"]
test_loader = loaders["test"]

train {'patients': 63, 'samples': 128, 'improving': 45, 'stable': 47, 'worsening': 36}
val {'patients': 11, 'samples': 13, 'improving': 7, 'stable': 2, 'worsening': 4}
test {'patients': 8, 'samples': 18, 'improving': 6, 'stable': 5, 'worsening': 7}
Excluded: {'input_not_positive': 1990, 'unreviewed_or_nontraining_label': 202}


In [74]:
from data_loader.cxr_backbone_dataloader import make_backbone_dataloaders

backbone_loaders = make_backbone_dataloaders(
    labels_csv=labels_path,
    reviews_csv=reviews_path,
    report_summary_csv=output_path / "report_summary.csv",
    split_csv=labelling_dir / "prognosis_patient_splits.csv",
    reports_root=path1,
    batch_size=16,
    image_size=224,
    num_workers=0,
    manifest_path=labelling_dir / "backbone_image_manifest.csv",
)

backbone_train_loader = backbone_loaders["train"]
backbone_val_loader = backbone_loaders["val"]
backbone_test_loader = backbone_loaders["test"]

train studies: 2008 patients: 354
val studies: 307 patients: 74
test studies: 392 patients: 77
Excluded: {'unusable_report': 150}


### A4.1. In 


In [77]:
import torch
from model.cxr_backbone import CXRBackbone, diagnosis_loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CXRBackbone(pretrained=True).to(device)

batch = next(iter(backbone_train_loader))
images = batch["image"].to(device)
targets = batch["targets"].to(device)

logits = model(images)
loss = diagnosis_loss(logits, targets)

print(logits.shape)  # [B, 5, 4]
print(loss.item())

3.6%

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to C:\Users\Zila/.cache\torch\hub\checkpoints\densenet121-a639ec97.pth


100.0%


torch.Size([16, 5, 4])
1.4523804187774658


In [79]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

Python: c:\Users\Zila\miniconda3\envs\longitudinal_progression\python.exe
PyTorch: 2.14.1+cpu
CUDA build: None
CUDA available: False
GPU count: 0


In [80]:
!nvidia-smi

Sat Oct  3 01:29:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.91                 Driver Version: 591.91         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5060 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   46C    P5              6W /   70W |     482MiB /   8151MiB |     19%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [78]:
from datetime import datetime
import torch

from model.cxr_backbone import CXRBackbone
from model.train_backbone import train_backbone

# Seed before initializing the model
torch.manual_seed(42)

model = CXRBackbone(pretrained=True)

run_dir = (
    project_dir / "training_outputs"
    / datetime.now().strftime("backbone_%Y%m%d_%H%M%S")
)

results = train_backbone(
    model=model,
    loaders=backbone_loaders,
    out_dir=run_dir,
    epochs=20,
    learning_rate=1e-4,
    weight_decay=1e-4,
)

Device: cpu | Epochs: 20 | Learning rate: 0.0001


KeyboardInterrupt: 

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    x = torch.ones(3, device="cuda")
    print("GPU calculation:", (x * 2).cpu())

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
GPU calculation: tensor([2., 2., 2.])
